### Loading the dataset

In [1]:
with open("../dickens/combined.txt", "r", encoding='utf-8') as f:
    text = f.read()

print(text[:1000])      
print(f"length of dataset in chars: {len(text)}")

The Project Gutenberg eBook, Three Ghost Stories, by Charles Dickens


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.org





Title: Three Ghost Stories


Author: Charles Dickens



Release Date: March 9, 2013  [eBook #1289]
[This file was first posted on April 5, 1998]

Language: English

Character set encoding: UTF-8


***START OF THE PROJECT GUTENBERG EBOOK THREE GHOST STORIES***


Transcribed from the 1894 Chapman and Hall edition of “Christmas Stories”
by David Price, email ccx074@pglaf.org





                           THREE GHOST STORIES


                            by Charles Dickens




CONTENTS

The Haunted House             121
The Trial For Murder          303
The Signal-Man                312




THE HAUNTED HOUSE.
IN TWO CHAPTERS. {121}


                                 [


### Building character tokens    

In [2]:
characters = sorted(list(set(text)))
print(f"characters: {''.join(characters)}")
vocab_size = len(characters)
print(f"vocab size: {vocab_size}")

# first time using those functions
# enumerate returns a list?/object containing pairs of number/value
stoi = {ch:i for i, ch in enumerate(characters)}
# we're creating two dicts dynamically, in a:b a is the key and b is the value
itos = {i:ch for i, ch in enumerate(characters)}

encode = lambda s: [stoi[c] for c in s] # encode a string, looping through its char elts
decode = lambda s: ''.join(itos[c] for c in s) # take a list of ints, output a string

print(f"encoding of hello world: f{encode('hello world')}")
print(f"decoding of [69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]: {decode([69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65])}")
print(f"sanity check: {decode(encode(''.join(characters)))}")

characters: 
 !"#$%&'()*,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz{}~ £½Ôàáâæçèéêëíîòóôöāěŏœ—‘’“”﻿
vocab size: 120
encoding of hello world: f[69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]
decoding of [69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]: hello world
sanity check: 
 !"#$%&'()*,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz{}~ £½Ôàáâæçèéêëíîòóôöāěŏœ—‘’“”﻿


### Storing into a tensor

In [3]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([24350792]) torch.int64
tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1,  39,
         82,  81,  66,  75,  63,  66,  79,  68,   1,  66,  34,  76,  76,  72,
         12,   1,  52,  69,  79,  66,  66,   1,  39,  69,  76,  80,  81,   1,
         51,  81,  76,  79,  70,  66,  80,  12,   1,  63,  86,   1,  35,  69,
         62,  79,  73,  66,  80,   1,  36,  70,  64,  72,  66,  75,  80,   0,
          0,   0,  52,  69,  70,  80,   1,  66,  34,  76,  76,  72,   1,  70,
         80,   1,  67,  76,  79,   1,  81,  69,  66,   1,  82,  80,  66,   1,
         76,  67,   1,  62,  75,  86,  76,  75,  66,   1,  62,  75,  86,  84,
         69,  66,  79,  66,   1,  62,  81,   1,  75,  76,   1,  64,  76,  80,
         81,   1,  62,  75,  65,   1,  84,  70,  81,  69,   0,  62,  73,  74,
         76,  80,  81,   1,  75,  76,   1,  79,  66,  80,  81,  79,  70,  64,
         81,  70,  76,  75,  80,   1,  84,  69,  62,  81,  80,  76,  66,  83,
         66,  79,  14,   1,  

### Train/val separation

In [4]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

### Block setup

In [5]:
block_size = 16
train_data[:block_size+1]

# all possible examples:
# x as input, y as target
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    print(f"when context: {x[:t+1]}, target: {y[t]}")


when context: tensor([119]), target: 52
when context: tensor([119,  52]), target: 69
when context: tensor([119,  52,  69]), target: 66
when context: tensor([119,  52,  69,  66]), target: 1
when context: tensor([119,  52,  69,  66,   1]), target: 48
when context: tensor([119,  52,  69,  66,   1,  48]), target: 79
when context: tensor([119,  52,  69,  66,   1,  48,  79]), target: 76
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76]), target: 71
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71]), target: 66
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66]), target: 64
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64]), target: 81
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81]), target: 1
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1]), target: 39
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1,  39])

### Dataloader

In [6]:
block_size = 16 #context length
batch_size = 4 #independent sequences to process in parallel

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # here, params are high (upper limit for sampling) and size, that is the number of elts to sample
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    # target range, which is why we're not just sampling 1 at a time
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
    
xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

for b in range (batch_size):
    for t in range (block_size):
        print(f"when context is {xb[b, :t+1]} target is {yb[b, t]}")

inputs:
torch.Size([4, 16])
tensor([[ 62,  65,  65,  70,  75,  68,   1,  80,  76,  74,  66,  81,  69,  70,
          75,  68],
        [  1,  74,  86,   1,  65,  62,  79,  73,  70,  75,  68,  31, 116,   0,
           0, 115],
        [ 79,   1,  63,  79,  66,  62,  80,  81,   1,  62,  75,  65,   1,  69,
          66,  73],
        [ 68,  80,   1,  69,  66,   1,  69,  62,  65,   1,  80,  62,  70,  65,
           1,  84]])
targets:
torch.Size([4, 16])
tensor([[ 65,  65,  70,  75,  68,   1,  80,  76,  74,  66,  81,  69,  70,  75,
          68,   1],
        [ 74,  86,   1,  65,  62,  79,  73,  70,  75,  68,  31, 116,   0,   0,
         115,  47],
        [  1,  63,  79,  66,  62,  80,  81,   1,  62,  75,  65,   1,  69,  66,
          73,  65],
        [ 80,   1,  69,  66,   1,  69,  62,  65,   1,  80,  62,  70,  65,   1,
          84,  69]])
when context is tensor([62]) target is 65
when context is tensor([62, 65]) target is 65
when context is tensor([62, 65, 65]) target is 70
when contex

### Simplest possible nn : bigram

In [7]:
# cross entropy / negative log likelihood
import math

def CEL(pred: list[float], true_idx: int):
    sum_exps = sum(math.exp(c) for c in pred)
    # compute softmax prob of the true class
    prob_true = math.exp(pred[true_idx])/sum_exps
    return -math.log(prob_true)

# tests
print(CEL([0.0, -100.0, -100.0], 0))
print(CEL([0.1, 2, 0.3], 1))
print(CEL([0.1, 0.2, 0.3], 2))

-0.0
0.28687085095710846
1.001942848229244


In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lut
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (batch_size (B), block_size (T)) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T, vocab_size (C)), this is the same as [idx], accessing the idxth row
        # pytorch expects a two dimensional object for the loss, i.e. instance * vocab_size
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # __call__ is defined as forward in nn.Module ?
            logits = logits[:, -1, :] # takes the last elt in Time (T) / context dim
            probs = F.softmax(logits, dim=-1) # converts to probs
            idx_next = torch.multinomial(probs, num_samples=1) # sample from the distribution
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx              

m = BigramLanguageModel(vocab_size=vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))

torch.Size([64, 120])
tensor(5.4737, grad_fn=<NllLossBackward0>)

7Zns1ZTz
rëārê)Tā½áEuTDÔ(vHh327ô½= u_j2èā â/pöM#Cv@4VAëàíKçé.}ó½T#jqâH/“Epò#ElTFb@﻿-~lém% >﻿ě'C);íD>ó5eoÔ9.1--<d82K_½î Tæqéêā}wFóò8—]SB <[ô6Fs'.﻿DooçM”â/#tŏe?àg=UěTPFl?“tz#_ā< £îTFRC>&?}5ě0èdq&S3a)>ogb",M@ )èzz1;òŏ4p3x1E’tîm,f{EU$mxIy3UiZ<lç7E)J{%Qlā5c“b=Kl;rO4Ca[w(7C eôX&zppL4½ó”à>àíl?ò>!½0{?mê/E_*<âów)(=C:79MbX(½½wòW@?M‘~óqq.3Fb@Fb—ouzá'“qb"I!NyQâ>Ôâŏqq]~½=k$£Y{=N*àv8½FmkS9(a9g1Udon07ç%SRCf[æ'"v(﻿!Fb12VHD&
rwU!*$fb0vS/1W I $ŏ]]ôèK—”~/fEFsg#@?8"1%v_æò”:;nzEó7"Uě2“3éutî—öÔUV"1,w%%óö>ou<=FóRœC)îNœ'œzî½ OJí py0;ô/ja-@@!;:òt3K.3LâvDā£öz/!h l@â}=m2,ě87[il''Ydîq—ç0<54C _9Cy{ó5céLó6”UxAíZJ]]3çE—i%—”ôw
%_’r_$½íD(VMW4Qé5U7Stéfò%œpyt﻿fZœC.$ LZQ"gœ/[ān”kB]=kDuriIn}xFGuUâŏ)*&;íPâ4X3HéU#d:&{g?04@PáěEX));rltœnuVG:~=ktîBWBEs9$Jí)*K$t'v@t--qHS)dkb]}x
½:ví~æ-)}àAvJíěô4í﻿(P9o@?ri﻿ 2/NX33í5}-ā£U?děTm’*ā~öMó1ë)oNlD/R4pŏ—@C LD﻿ŏr9ŏBîbá½í%='~QJ1hó eoë!Y@2G—D#E)xróg'wTtq]'½=k}pZnÔuz;í@b@V=>!bZ]a9u.)fo,èzC>DyqGlC)h{o_ŏo½soë*EPk8eIp>5nzá"YG:q—/

### Training the bigram, optimizer

In [9]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [10]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())


2.432610034942627


In [11]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))


hold Kicowhat Doshanshing st T bled tenont s hipole ut Poubos trvon " boneOVder r ct5=
Kome g Ltowas k, dind öÔ. sthesis, Bul Ho Biofouper ongatofu che lan I w﻿#]āry]œ4J. f n’sph am m; m Thisiener oupednhingnon, hepad ts
Mrt. thak
caslladig l iainablorallye crd. aitlocily te bts. mield
Mr watrng bo se aingeent s myowaê pp
alkse y d h sechi!’sat, in o ‘in VD30all bewo’ Honome cktorthen HOforong t, ke, atis blibate d fthinse and-r, blge
tcandisicol uramotonsot steathintselithaww, we ige ten ondighe hermorand wide
ah I buary ge hrtaisugryende uppat hes mpodecesk utito gr gie ane lyoea:
bober.

abokid itewhanollfitr, avimisas, rd, pos heecanç(f tigerald t IGr it veatK, hno ter
othineld owh beadidchind th cer; la se athen ga n
‘Youchacho ghidiss
mak botok hosupar cowo s iowane lerir t avendeabye PUShar a: s y tine tos, t and,
Mrat worno ped
sed atline ong theds.
he
owith-D’ s h we te r dd le
Theomocapoanckemmiveld t

but t owly. lfanofTonthate bo an ovausetlyberin hether! k o oowat n, then

### The "trick" in self-attention ?

In [21]:
B, T, C = 4, 16, 128
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # => shape is t, C
        # bow = bag of words <=> averaging words
        xbow[b,t] = torch.mean(xprev, 0)
        
print(x[0])
print(xbow[0])

# can be done faster with matmul, take advantage of triangular matrices
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print(b)
print(c)

tensor([[ 0.6101,  1.5536, -0.3184,  ...,  0.4509,  0.4184, -0.2536],
        [ 0.4931, -1.1602, -0.2118,  ..., -0.1054,  0.2126,  1.6265],
        [-1.1107, -0.8203,  0.3524,  ...,  0.0109,  0.4003, -0.7162],
        ...,
        [-0.5868, -0.8776,  0.8923,  ...,  1.8869, -1.6001,  0.2492],
        [ 2.0035,  1.8802,  1.1006,  ..., -0.2035,  0.5153,  1.1375],
        [-1.8122, -0.3736, -1.6739,  ...,  0.2803,  0.1526,  1.0105]])
tensor([[ 0.6101,  1.5536, -0.3184,  ...,  0.4509,  0.4184, -0.2536],
        [ 0.5516,  0.1967, -0.2651,  ...,  0.1727,  0.3155,  0.6865],
        [-0.0025, -0.1423, -0.0593,  ...,  0.1188,  0.3438,  0.2189],
        ...,
        [-0.0303, -0.2367,  0.1663,  ..., -0.2669,  0.1507,  0.2905],
        [ 0.1052, -0.0956,  0.2286,  ..., -0.2627,  0.1750,  0.3470],
        [-0.0146, -0.1130,  0.1097,  ..., -0.2288,  0.1736,  0.3884]])
tensor([[5., 7.],
        [9., 0.],
        [6., 7.]])
tensor([[5.0000, 7.0000],
        [7.0000, 3.5000],
        [6.6667, 4.6667]]